# graphArt_LM: the two-step model with the full-rules prompts

Fine-tunes `Qwen2.5-Coder-1.5B-Instruct` with a fresh LoRA on the two-step format (nodes
with their source lines, then edges with the variables they are read from), like the last
run, but with the full-rules prompts: the brief ones left out the rule that a target is the
attribute path, and the last adapter wrote `_forward` into module paths. To pay for the
rules, tiers 1-4 are cut to every eighth program.

Then scores it on the same 40 test programs and the 21 torchvision samples, step 2 always
given the model's own step-1 nodes. About 2.4 hours, capped at 2.5.

**Before running**: *GPU T4 x2* (one is used), *Internet on*. No inputs needed. Then
*Save Version -> Save & Run All (Commit)*.

**When it finishes**, download `graphart_finetune_two_step_full.zip` from the *Output* tab:
the results and the adapter.

Generated by `scripts/make_kaggle_notebook.py --finetune-two-step-full`. Edit that script, not this file.

In [ ]:
# BASE_MODEL   tuned from scratch (a fresh LoRA), like every earlier session.
# ADAPTER      its name; results land in results/hf__<ADAPTER>__two_step*.jsonl.
# TRAIN_ARGS   --two-step --full-prompts: two examples per program (nodes, then edges),
#              with the full-rules prompts. train/8 is every eighth tier 1-4 program (40 a
#              tier); all of tiers 5-6; val/8 and half of large_val. Longest example 7022.
# FALLBACK_MAX_LEN  if the longest examples do not fit in GPU memory, train with this
#              --max-len instead and leave out the few longer examples.
# MEMORY_CHECK forward + backward on this many of the longest examples, before anything else.
# EVALS        (data set, reply cap per call), each program in two calls with the same
#              full prompts: the 40 stratified test programs and the 21 torchvision samples.
# EVAL_RESERVE_HOURS  kept back from training for EVALS; training ~100 min, and the time
#              guard stops it early (keeping the best checkpoint) if it would eat into this.
# VALIDATE_ONLY True: only check the code arrived intact (works on a CPU session).
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; ADAPTER = "qwen2.5-coder-1.5b-graphart-lora-two-step-full"; EPOCHS = 1.0; TRAIN_ARGS = ["--two-step", "--full-prompts", "--train-split", "train/8,large_train", "--val-split", "val/8,large_val/2", "--max-len", "8192", "--eval-steps", "40"]; FALLBACK_MAX_LEN = 6144; MEMORY_CHECK = 6; SMOKE_STEPS = 8; EVALS = [("test", 3000), ("torchvision", 4500)]; EVAL_RESERVE_HOURS = 0.8; TIME_BUDGET_HOURS = 2.5; EVAL_ARGS = []; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "7273a2de2ded751d48515ebb258bbbf318a035a6828bc30216ab478acff09ff4"
BUNDLE = (
    "..."
    "AADcNYFOFBc7MAMAAAAABFla"
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:xz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install the pinned libraries

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PINS = {'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0'}

def pinned(package, wanted):
    try:
        return version(package) == wanted
    except PackageNotFoundError:
        return False

missing = [f"{p}=={v}" for p, v in PINS.items() if not pinned(p, v)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# peft probes two optional quantisation libraries when it builds LoRA layers, and
# raises -- rather than skipping them -- if they are installed but older than it
# supports. Kaggle ships torchao 0.10 (peft wants >= 0.16), which killed a run. We
# use neither, so remove them if present.
for pkg in ("torchao", "gptqmodel"):
    try:
        print(f"removing {pkg} {version(pkg)} (peft rejects old versions; not used here)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], check=True)
    except PackageNotFoundError:
        pass

subprocess.run([sys.executable, "-c", "import torch, transformers, peft; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "'| transformers', transformers.__version__, '| peft', peft.__version__)"])

## 3. Download the base model

In [ ]:
# Its own step with a time limit: a stalled download inside training would otherwise
# sit until the session cap. ~3 GB; a few minutes on Kaggle's network.
FETCH = ("from huggingface_hub import snapshot_download; "
         f"print('downloaded to', snapshot_download({BASE_MODEL!r}))")
for attempt in (1, 2):
    try:
        subprocess.run([sys.executable, "-c", FETCH], check=True, timeout=1800)
        break
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print(f"download attempt {attempt} failed: {exc}", flush=True)
else:
    raise SystemExit("could not download the base model -- stopping before any training")

## 4. Check the data, then the GPU memory

In [ ]:
import zipfile

GPU_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="0", HF_HUB_OFFLINE="1", PYTHONUNBUFFERED="1")
ZIP_NAME = "graphart_finetune_two_step_full.zip"

def remaining():
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

def train(*extra):
    return subprocess.run([sys.executable, "scripts/train_lora.py", "--base", BASE_MODEL,
                           *TRAIN_ARGS, *extra], env=GPU_ENV).returncode

def package():
    with zipfile.ZipFile(WORK / ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
        for root in (REPO / "results", REPO / "lora" / ADAPTER):
            for p in root.rglob("*") if root.exists() else []:
                if p.is_file() and "checkpoints" not in p.parts:
                    z.write(p, p.relative_to(REPO))

def with_max_len(args, n):
    out = list(args)
    out[out.index("--max-len") + 1] = str(n)
    return out + ["--drop-over-max"]

if train("--out", "lora/check", "--check-only") != 0:
    raise SystemExit("the data check failed -- see the traceback above")
code = train("--out", "lora/check", "--memory-check", str(MEMORY_CHECK))
if code == 5:                                 # train_lora.py: out of memory
    print(f"the longest examples do not fit -- falling back to --max-len {FALLBACK_MAX_LEN}", flush=True)
    TRAIN_ARGS = with_max_len(TRAIN_ARGS, FALLBACK_MAX_LEN)
    code = train("--out", "lora/check", "--memory-check", str(MEMORY_CHECK))
if code != 0:
    raise SystemExit(f"the memory check failed (exit {code}) -- see the log above")

## 5. Smoke run: is it learning?

In [ ]:
code = train("--out", "lora/smoke", "--smoke-steps", str(SMOKE_STEPS))
if code in (3, 4):                            # not learning / NaN: maybe fp16, retry in fp32
    print(f"smoke run failed in fp16 (exit {code}) -- retrying in fp32", flush=True)
    code = train("--out", "lora/smoke", "--smoke-steps", str(SMOKE_STEPS), "--fp32")
    if code == 0:
        TRAIN_ARGS = [*TRAIN_ARGS, "--fp32"]
if code != 0:
    raise SystemExit(f"the smoke run failed (exit {code}) -- stopping before the full run")

## 6. Train

In [ ]:
hours = max(remaining() / 3600 - EVAL_RESERVE_HOURS, 0.25)
print(f"training for at most {hours:.2f} h", flush=True)
if train("--out", f"lora/{ADAPTER}", "--epochs", str(EPOCHS), "--time-budget-hours", f"{hours:.2f}") != 0:
    raise SystemExit("training failed -- see the log above")
print((REPO / "lora" / ADAPTER / "train_summary.json").read_text())
package()

## 7. Score it

In [ ]:
adapter = REPO / "lora" / ADAPTER
for split, cap in EVALS:
    if remaining() < 600:
        print(f"time budget spent -- skipping {split}")
        continue
    cmd = [sys.executable, "-m", "pipeline.run_two_step", "--backend", "hf", "--model", str(adapter),
           *[], "--split", split, *(["--max-tokens", str(cap)] if cap else []),
           *(["--stratify", "10"] if split == "test" else []), *EVAL_ARGS]
    print(f"\n##### {split}   ({remaining() / 3600:.1f} h left)", flush=True)
    try:
        subprocess.run(cmd, env=GPU_ENV, timeout=remaining())
    except subprocess.TimeoutExpired:
        print("stopped by the time budget -- the finished replies are kept")
    package()
for meta in sorted((REPO / "results").glob("*.meta.json")):
    m = json.loads(meta.read_text())
    print(f"{meta.stem:75s} n={m['n']} failed={m['failed']}")
print(f"done in {(time.time() - START) / 3600:.1f} h -- download {ZIP_NAME}")